# Local Invoice OCR on Google Colab
Select **Runtime > Change runtime type > T4 GPU**, then run every cell in order. GPU runtimes install CUDA Paddle; CPU runtimes use CPU Paddle. Public repositories need no token; private repositories need the GITHUB_TOKEN Colab secret. AI parsing is optional and disabled by default for faster results. The first upload downloads and loads OCR models.

In [ ]:
#@title 1. Download the project
import io, os, pathlib, shutil, urllib.error, urllib.request, zipfile
from google.colab import userdata
PROJECT_DIR = pathlib.Path('/content/OCR')
ARCHIVE_URL = 'https://api.github.com/repos/ubaid-148/OCR/zipball/main'
headers = {'Accept': 'application/vnd.github+json', 'User-Agent': 'OCR-Colab-Setup'}
try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = None
if github_token:
    headers['Authorization'] = f'Bearer {github_token}'
request = urllib.request.Request(ARCHIVE_URL, headers=headers)
try:
    with urllib.request.urlopen(request, timeout=120) as response:
        archive_bytes = response.read()
except urllib.error.HTTPError as error:
    raise RuntimeError(f'GitHub download failed ({error.code}). Make the repo public or grant GITHUB_TOKEN read access.') from error
extract_root = pathlib.Path('/content/ocr-download')
shutil.rmtree(extract_root, ignore_errors=True)
shutil.rmtree(PROJECT_DIR, ignore_errors=True)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    archive.extractall(extract_root)
extracted = next(path for path in extract_root.iterdir() if path.is_dir())
shutil.move(str(extracted), str(PROJECT_DIR))
os.chdir(PROJECT_DIR)
print('Private project ready at', PROJECT_DIR)

In [ ]:
#@title 2. Install dependencies and verify the OCR device
import os, pathlib, shutil, subprocess, sys
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'tesseract-ocr', 'tesseract-ocr-eng', 'tesseract-ocr-ara', 'tesseract-ocr-urd', 'ghostscript', 'unpaper', 'pngquant', 'zstd'], check=True)
gpu_runtime = bool(shutil.which('nvidia-smi')) and subprocess.run(['nvidia-smi', '-L'], capture_output=True).returncode == 0
# Keep OCR packages separate from Colab's preinstalled CUDA PyTorch.
OCR_ENV_DIR = pathlib.Path('/content/ocr-runtime')
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(OCR_ENV_DIR)], check=True)
OCR_PYTHON = str(OCR_ENV_DIR / 'bin' / 'python')
# pip --python bootstraps pip even in a venv created without it.
ocr_pip = [sys.executable, '-m', 'pip', '--python', OCR_PYTHON]
subprocess.run([*ocr_pip, 'install', '-q', '--upgrade', 'pip'], check=True)
# ModelScope imports torch; its CPU build avoids a second CUDA/NCCL stack.
subprocess.run([*ocr_pip, 'install', '-q', 'torch==2.9.1+cpu', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
# CPU and GPU Paddle share a module: install exactly one distribution.
subprocess.run([*ocr_pip, 'uninstall', '-y', 'paddlepaddle', 'paddlepaddle-gpu'], check=True)
requirements = [line.strip() for line in pathlib.Path('requirements.txt').read_text().splitlines() if line.strip() and not line.strip().startswith('paddlepaddle')]
subprocess.run([*ocr_pip, 'install', '-q', *requirements], check=True)
command = [*ocr_pip, 'install', '-q']
if gpu_runtime:
    command += ['paddlepaddle-gpu==3.3.1', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/']
else:
    command += ['paddlepaddle==3.3.1']
subprocess.run(command, check=True)
os.environ['OCR_DEVICE'] = 'gpu:0' if gpu_runtime else 'cpu'
os.environ.pop('OCR_PYTHON_EXE', None)
# Check in a fresh process so rerunning this cell cannot reuse an old Paddle import.
os.environ.setdefault('FLAGS_use_mkldnn', '0')
verification = subprocess.run(
    [OCR_PYTHON, '-u', str(PROJECT_DIR / 'check_ocr_runtime.py')],
    cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, errors='replace',
)
verification_log = pathlib.Path('/tmp/ocr-runtime-check.log')
verification_log.write_text(verification.stdout, encoding='utf-8')
print(verification.stdout, flush=True)
if verification.returncode:
    raise RuntimeError(
        f'OCR runtime verification failed (exit {verification.returncode}). '
        f'Full log: {verification_log}. Copy the error below:\n\n'
        + verification.stdout[-12000:]
    )
print('Dependencies ready. First upload loads OCR models; later uploads reuse them.')


In [ ]:
#@title 3. Start local AI (Ollama)
USE_LOCAL_AI = False #@param {type:"boolean"}
OLLAMA_MODEL = 'qwen2.5:3b' #@param {type:"string"}
import os, shutil, subprocess, time, urllib.request
os.environ['OLLAMA_MODEL'] = OLLAMA_MODEL
os.environ['USE_LOCAL_AI'] = str(USE_LOCAL_AI).lower()
os.environ['OLLAMA_TIMEOUT_SECONDS'] = '60'
os.environ.pop('OLLAMA_URL', None)
if USE_LOCAL_AI:
    if not shutil.which('ollama'):
        ollama_archive = '/tmp/ollama-linux-amd64.tar.zst'
        print('Downloading Ollama...')
        urllib.request.urlretrieve('https://ollama.com/download/ollama-linux-amd64.tar.zst', ollama_archive)
        subprocess.run(['tar', '--zstd', '-xf', ollama_archive, '-C', '/usr'], check=True)
        if not shutil.which('ollama'):
            raise RuntimeError('Ollama archive extracted but the executable was not found')
    ollama_log = open('/tmp/ollama.log', 'w')
    ollama_process = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError('Ollama did not start. Check /tmp/ollama.log')
    subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True)
    print('Ollama ready:', OLLAMA_MODEL)
else:
    os.environ['OLLAMA_URL'] = 'http://127.0.0.1:1/api/chat'
    print('Local AI disabled; deterministic spatial fallback will be used.')

In [ ]:
#@title 4. Start OCR web application
import os, subprocess, sys, time, urllib.request
os.chdir('/content/OCR')
if 'OCR_PYTHON' not in globals():
    raise RuntimeError('Run dependency setup (cell 2) first.')
if 'ocr_process' in globals() and ocr_process.poll() is None:
    ocr_process.terminate()
    ocr_process.wait(timeout=15)
ocr_log = open('/tmp/ocr-web.log', 'w')
ocr_process = subprocess.Popen([OCR_PYTHON, '-u', 'ocr_web.py'], stdout=ocr_log, stderr=subprocess.STDOUT, env=os.environ.copy())
for _ in range(120):
    if ocr_process.poll() is not None:
        print(open('/tmp/ocr-web.log').read())
        raise RuntimeError('OCR server exited during startup')
    try:
        response = urllib.request.urlopen('http://127.0.0.1:8765/', timeout=2)
        if response.status == 200:
            break
    except Exception:
        time.sleep(1)
else:
    print(open('/tmp/ocr-web.log').read())
    raise RuntimeError('OCR web application did not start')
print('OCR application is ready.')

In [ ]:
#@title 5. Open the application
from google.colab import output
output.serve_kernel_port_as_iframe(8765, height='700')

Upload a PDF in the embedded application. Models are reused after the first upload. Balanced mode skips AI when the spatial result passes checks; select Fast to skip AI entirely (review fields marked needs_review). JSON timings_seconds separates model loading, OCR, and parsing. Set USE_LOCAL_AI=False in cell 3 to disable AI for all uploads. Colab storage and processes are temporary; rerun the notebook after a runtime reset. If startup fails, inspect `!/tmp/ocr-web.log` or `!/tmp/ollama.log`.